In [1]:
%pip install -q transformers huggingface_hub
import math
import torch
import transformers

import numpy as np
import torch.nn as nn
import torch.nn.functional as F


# Build-a-transformer

In this section, you will implement a transformer language model layer by layer, then use it to generate (hopefully) coherent text.

To understand how these layers work, please check out our guide to transformers from [nlp course for you -> transformers](https://lena-voita.github.io/nlp_course/seq2seq_and_attention.html#transformer_intro).


First, we download pre-trained weights for the [GPT2 model by OpenAI](https://openai.com/research/better-language-models) - a prominent model from 2019.



Idea & code by: Ilya Beletsky

In [2]:
from huggingface_hub import hf_hub_download
state_dict = torch.load(hf_hub_download("gpt2", filename="pytorch_model.bin"))

for key, value in tuple(state_dict.items()):
    if key.startswith('h.') and key.endswith('.weight') and value.ndim == 2:
        value.transpose_(1, 0)  # <-- for compatibility with modern PyTorch modules
    if key.startswith('h.') and key.endswith('.attn.bias') and value.ndim == 4:
        state_dict.pop(key)  # <-- triangular binar masks, not needed in this code

print('Weights:', repr(sorted(state_dict.keys()))[:320], '...')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/548M [00:00<?, ?B/s]

<ipython-input-2-531e139fa06e>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(hf_hub_download("gpt2", filename="pytorch_model.bin"))


Weights: ['h.0.attn.c_attn.bias', 'h.0.attn.c_attn.weight', 'h.0.attn.c_proj.bias', 'h.0.attn.c_proj.weight', 'h.0.ln_1.bias', 'h.0.ln_1.weight', 'h.0.ln_2.bias', 'h.0.ln_2.weight', 'h.0.mlp.c_fc.bias', 'h.0.mlp.c_fc.weight', 'h.0.mlp.c_proj.bias', 'h.0.mlp.c_proj.weight', 'h.1.attn.c_attn.bias', 'h.1.attn.c_attn.weight', 'h.1. ...


In the next few cells, we shall implement the model layer by layer to make use of those weights.

As you might recall, transformers contain two main layer types: attention and fully-connected layers.

The fully connected layers are by far easier to understand, so we shall begin there:

Please implement fully-connected layer __without residual or layer normalization__ (we'll add those in a bit).

## Implement the fully connected part of the model (3 points)

In [3]:
class GeLUThatWasUsedInGPT2(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x ** 3)))

class FullyConnected(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.c_fc = nn.Linear(dim, 4  * dim)
        self.gelu = GeLUThatWasUsedInGPT2()
        self.c_proj = nn.Linear(4 * dim, dim)

    def forward(self, x):
        '''x.shape = [batch_size, seq_length, dim]'''

        # <YOUR CODE HERE - COMPUTE LAYER OUTPUTS>
        x = self.c_fc(x) # [batch_size, seq_length, 4 * dim]
        x = self.gelu(x) # [batch_size, seq_length, 4 * dim]
        x = self.c_proj(x) # [batch_size, seq_length, dim]
        # </YOUR CODE HERE - COMPUTE LAYER OUTPUTS>

        return x

___Test FullyConnected___

In [4]:
# test
mlp = FullyConnected(8)

a = torch.randint(0, 10, [1, 8], dtype=torch.float)
print(a, a.shape)

with torch.no_grad():
  fc_a = mlp(a)
  print(fc_a, fc_a.shape)

tensor([[4., 1., 2., 9., 4., 7., 1., 5.]]) torch.Size([1, 8])
tensor([[ 0.3250,  1.6025,  0.9996,  1.5824, -2.2923, -0.5402,  0.8936,  0.1537]]) torch.Size([1, 8])


Now, let's test that it works with GPT-2 weights:

In [5]:
mlp = FullyConnected(dim=768)
mlp.load_state_dict({'c_fc.weight': state_dict['h.0.mlp.c_fc.weight'],
                     'c_fc.bias': state_dict['h.0.mlp.c_fc.bias'],
                     'c_proj.weight': state_dict['h.0.mlp.c_proj.weight'],
                     'c_proj.bias': state_dict['h.0.mlp.c_proj.bias']})

torch.manual_seed(1337)
x = torch.randn(1, 2, 768)  # [batch_size, sequence_length, dim]
checksum = torch.sum(mlp(x) * x)
assert abs(checksum.item() - 1282.3315) < 0.1, "layer outputs do not match reference"
assert torch.allclose(mlp(x[:, (1, 0), :])[:, (1, 0), :], mlp(x)), "mlp must be permutation-invariant"
print("Seems legit!")

Seems legit!


Now, let's get to attention layers.

Since GPT-2 needs to generate text from left to right, each generated token can only attend to tokens on the left (and itself). This kind of attention is called "Masked" self-attention, because it hides tokens to the right.

Below is a demonstration of how it could be implemented inefficiently. Please use lecture materials and the code below to figure out the proper implementation.

In [6]:
class MaskedSelfAttentionSlow(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.c_attn = nn.Linear(dim, dim * 3)  # query + key + value, combined
        self.c_proj = nn.Linear(dim, dim)  # output projection
        self.dim = dim
        self.num_heads = num_heads
        self.head_size = dim // num_heads

    def forward(self, x):
        q, k, v = self.c_attn(x).split(dim=-1, split_size=self.dim) # [seq_len, dim_model]
        assert q.shape == k.shape == v.shape == x.shape, "q, k and v must have the same shape as x"

        print(q.shape)
        print(self.dim, self.num_heads, self.head_size)


        # Note: this is an inefficient implementation that uses a for-loop.
        # To get the full grade during homework, please re-implement this code:
        # 1) do not use for-loops (or other loops). Compute everything in parallel with vectorized operations
        # 2) do not use F.scaled_dot_product_attention - write your own attention code using basic PyTorch ops

        # Add a dimension equal to the number of heads
        size_1 = q.size(0)
        q = q.view(size_1, self.num_heads, self.head_size).transpose(0, 1) # [seq_len, num_heads, head_size] >> [num_heads, seq_len, head_size]
        k = k.view(size_1, self.num_heads, self.head_size).transpose(0, 1) # [seq_len, num_heads, head_size] >> [num_heads, seq_len, head_size]
        v = v.view(size_1, self.num_heads, self.head_size).transpose(0, 1) # [seq_len, num_heads, head_size] >> [num_heads, seq_len, head_size]


        # Create Mask
        mask_size_q = q.size(-2) # [seq_len]
        mask_size_k = k.size(-2) # [seq_len]

        attn_bias = torch.zeros(mask_size_q, mask_size_k, dtype=q.dtype) # [seq_len, seq_len]
        temp_mask = torch.ones(mask_size_q, mask_size_k, dtype=torch.bool).tril(diagonal=0) # [seq_len, seq_len]
        attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf")) # [seq_len, seq_len]
        attn_bias.to(q.dtype) # [seq_len, seq_len]

        # Masked-self-attention
        q_kt = q @ k.transpose(1, 2) # [num_heads, seq_len, head_size] @ [num_heads, head_size, seq_len] >> [num_heads, seq_len, seq_len]
        q_kt_masked = q_kt + attn_bias # [num_heads, seq_len, seq_len]+ [seq_len, seq_len] >> [num_heads, seq_len, seq_len]
        attention_scores = F.softmax(q_kt_masked / (self.head_size ** 0.5), dim=-1) # [num_heads, seq_len, seq_len]
        attention_output = attention_scores @ v # [num_heads, seq_len, seq_len] @ [num_heads, seq_len, head_size] >> [num_heads, seq_len, head_size]

        # concat heads and returning the original dimension
        attention_output = torch.cat(attention_output.unbind(dim=0), dim=1) # [num_heads, seq_len, head_size] >> [seq_len, head_size * num_heads]
        attention_output = self.c_proj(attention_output)

        return attention_output

In [7]:
AI = MaskedSelfAttentionSlow(dim=64, num_heads=2)

input_m = torch.randint(0, 10, [10, 64], dtype=torch.float)
# self.dim, self.num_heads, self.head_size
AI.forward(input_m).size()

torch.Size([10, 64])
64 2 32


torch.Size([10, 64])

In [8]:
seq_len, num_heads, head_size = 4, 2, 32
a = torch.randint(0, 10, [4, 64], dtype=torch.float)
print(a.size())
q = a.view(seq_len, num_heads, head_size).transpose(0, 1)
q.size()

torch.Size([4, 64])


torch.Size([2, 4, 32])

## Implement masked self-attention __without layernorm or residual connections.__ (5 points)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedSelfAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.c_attn = nn.Linear(dim, dim * 3)  # query + key + value, combined
        self.c_proj = nn.Linear(dim, dim)  # output projection
        self.dim = dim
        self.num_heads = num_heads
        self.head_size = dim // num_heads
        self.scale = self.head_size ** -0.5

    def forward(self, x):

        CNT_BATCH = x.size(0)
        BATCH_SIZE = x.size(1)

        ### Compute query, key, and value matrices
        q, k, v = self.c_attn(x).split(dim=-1, split_size=self.dim) # [cnt_batch, batch_size, dim_model]
        assert q.shape == k.shape == v.shape == x.shape, "q, k and v must have the same shape as x"

        ### Reshape for multi-head attention
        q = q.view(CNT_BATCH, BATCH_SIZE, self.num_heads, self.head_size).transpose(1, 2) # [cnt_batch, batch_size, dim_model] >> [cnt_batch, cnt_head, batch_size, head_size]
        k = k.view(CNT_BATCH, BATCH_SIZE, self.num_heads, self.head_size).transpose(1, 2) # [cnt_batch, batch_size, dim_model] >> [cnt_batch, cnt_head, batch_size, head_size]
        v = v.view(CNT_BATCH, BATCH_SIZE, self.num_heads, self.head_size).transpose(2, 1) # [cnt_batch, batch_size, dim_model] >> [cnt_batch, cnt_head, batch_size, head_size]

        ### Compute scaled dot-product attention, Apply the causal mask
        # Create Mask
        mask_size_q = q.size(-2) # [batch_size]
        mask_size_k = k.size(-2) # [batch_size]

        attn_bias = torch.zeros(mask_size_q, mask_size_k, dtype=q.dtype) # [batch_size, batch_size]
        temp_mask = torch.ones(mask_size_q, mask_size_k, dtype=torch.bool).tril(diagonal=0) # [batch_size, batch_size]
        attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf")) # [batch_size, batch_size]
        attn_bias.to(q.dtype) # [batch_size, batch_size]

        # Masked-self-attention
        q_kt = q @ k.transpose(2, 3) # [cnt_batch, cnt_head, batch_size, head_size] @ [cnt_batch, cnt_head, head_size, batch_size] >> [cnt_batch, cnt_head, batch_size, batch_size]
        q_kt_masked = q_kt + attn_bias # [cnt_batch, cnt_head, batch_size, batch_size] + [batch_size, batch_size] >> [cnt_batch, cnt_head, batch_size, batch_size]


        # Apply softmax to get attention probabilities
        attention_scores = F.softmax(q_kt_masked / (self.head_size ** 0.5), dim=-1) # [cnt_batch, cnt_head, batch_size, batch_size]

        # # Compute the weighted sum of values
        attention_output = attention_scores @ v # [cnt_batch, cnt_head, batch_size, batch_size] @ [cnt_batch, cnt_head, batch_size, head_size] >> [cnt_batch, cnt_head, batch_size, head_size]

        # # Return contectual embeddings
        attention_output = torch.cat(attention_output.unbind(dim=1), dim=2) ## [cnt_batch, cnt_head, batch_size, head_size] >> [cnt_batch, batch_size, head_size * cnt_head]
        attention_output = self.c_proj(attention_output) # [cnt_batch, batch_size, head_size * cnt_head]

        return attention_output

In [12]:
# test
x = torch.randn(8, 16, 768)  # [batch_size, sequence_length, dim]
print(x.size())

MSA = MaskedSelfAttention(dim=768, num_heads=12)

q_kt_masked = MSA(x)
print(q_kt_masked.size())

torch.Size([8, 16, 768])
torch.Size([8, 16, 768])


Test that it works

In [13]:
attn = MaskedSelfAttention(dim=768, num_heads=12)
attn.load_state_dict({'c_attn.weight': state_dict['h.0.attn.c_attn.weight'],
                      'c_attn.bias': state_dict['h.0.attn.c_attn.bias'],
                      'c_proj.weight': state_dict['h.0.attn.c_proj.weight'],
                      'c_proj.bias': state_dict['h.0.attn.c_proj.bias']})

torch.manual_seed(1337)
x = torch.randn(1, 10, 768)  # [batch_size, sequence_length, dim]
checksum = torch.sum(attn(x) * x)
assert abs(checksum.item() - 2703.6772) < 0.1, "layer outputs do not match reference"
assert not torch.allclose(attn(x[:, (1, 0), :])[:, (1, 0), :], attn(x[:, (0, 1), :])), "masked attention must *not* be permutation-invariant"
print("It works!")

It works!


We can now combine attention and MLP to build the full transformer layer:

![img](https://i.imgur.com/1sq2vHO.png)

## Bring it together (2 points)

In [14]:
class TransformerLayer(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(dim)
        self.attn = MaskedSelfAttention(dim, num_heads)
        self.ln_2 = nn.LayerNorm(dim)
        self.mlp = FullyConnected(dim)

    def forward(self, x):

        # <YOUR CODE - apply attention, mlp and layer normalization as shown in figure above>
        x_sa = self.ln_1(x)
        x_sa = self.attn(x_sa)
        x = x + x_sa
        x_fc = self.ln_2(x)
        x_fc = self.mlp(x_fc)
        x = x + x_fc


        return x

In [15]:
layer = TransformerLayer(dim=768, num_heads=12)
layer.load_state_dict({k[5:]: v for k, v in state_dict.items() if k.startswith('h.10.')})
assert abs(torch.sum(layer(x) * x).item() - 9874.7383) < 0.1
print("Good job!")

Good job!


In [16]:
class GPT2(nn.Module):
    def __init__(self, vocab_size: int, dim: int, num_heads: int, num_layers: int, max_position_embeddings: int = 1024):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, dim)  # token embeddings
        self.wpe = nn.Embedding(max_position_embeddings, dim)  # position embeddings
        self.ln_f = nn.LayerNorm(dim)   # final layer norm - goes after all transformer layers, but before logits

        self.h = nn.Sequential(*(TransformerLayer(dim, num_heads) for layer in range(num_layers)))

    def forward(self, input_ids):
        # input_ids.shape: [batch_size, sequence_length], int64 token ids
        position_ids = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)

        token_embeddings = self.wte(input_ids)
        position_embeddings = self.wpe(position_ids)
        full_embeddings = token_embeddings + position_embeddings

        transformer_output = self.h(full_embeddings)
        transformer_output_ln = self.ln_f(transformer_output)

        # final layer: we predict logits by re-using token embeddings as linear weights
        output_logits = transformer_output_ln @ self.wte.weight.T
        return output_logits


In [17]:
tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2', add_prefix_space=True)
model = GPT2(vocab_size=50257, dim=768, num_heads=12, num_layers=12)
model.load_state_dict(state_dict)

input_ids = tokenizer("A quick", return_tensors='pt')['input_ids']

predicted_logits = model(input_ids)
most_likely_token_id = predicted_logits[:, -1].argmax().item()

print("Prediction:", tokenizer.decode(most_likely_token_id))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Prediction:  look


In [57]:
text = "Can you tell me something about racing Formula 1? "
tokens = tokenizer.encode(text)
print(end=tokenizer.decode(tokens))
line_length = len(tokenizer.decode(tokens))

for i in range(500):
    # Predict logits with your model
    with torch.no_grad():
        logits = model(torch.as_tensor([tokens]))

    # Sample with probabilities
    p_next = torch.softmax(logits[0, -1, :], dim=-1).data.cpu().numpy()
    next_token_index = np.random.choice(len(p_next), p=p_next)

    tokens.append(int(next_token_index))
    print(end=tokenizer.decode(tokens[-1]))
    line_length += len(tokenizer.decode(tokens[-1]))
    if line_length > 120:
      line_length = 0
      print()

 Can you tell me something about racing Formula 1?  You won by alone. Why didn't we even want to? Who would play for Team
  now that Ferrari has taken great care of the comparable drivers pool?  What job did one have here except to race the other
 puppies, schools, forces, specifiers, etc.
JGW: History tells us that Formula 1 is the biggest story in American sports history
.  It helped convince baseball to change from an MLB to a NFL. It helped the milk-cart race leaders to set straight straw
 men and old quadriplegics on furloughs without regard to the sport's style or 'abolition of politics,'" says Kyle Klassic
. "You can't control it from the top and yet 10 years later it still seems like a much-heartened sport in the NFL. -- Hyper
baus Construction magazine; Former Rich Takes about what it was like to participate in the 18* maddeningly  brief schooly
ard kick*, which includes carcinogenic skids and splinters and stench from people's teeth ; �� a victory when such Nazis burned
 up r

In [ ]:
text = "The Fermi paradox "
tokens = tokenizer.encode(text)
print(end=tokenizer.decode(tokens))
line_length = len(tokenizer.decode(tokens))

for i in range(500):
    # Predict logits with your model
    with torch.no_grad():
        logits = model(torch.as_tensor([tokens]))

    # Sample with probabilities
    p_next = torch.softmax(logits[0, -1, :], dim=-1).data.cpu().numpy()
    next_token_index = np.random.choice(len(p_next), p=p_next)

    tokens.append(int(next_token_index))
    print(end=tokenizer.decode(tokens[-1]))
    line_length += len(tokenizer.decode(tokens[-1]))
    if line_length > 120:
      line_length = 0
      print()



 The Fermi paradox Ã�� Italian in Ramon Campantoni, who thought those wise heiresses might have destroyed chastity... Perhaps
 one of the worst paradoxes of all is that there is such a good twilight in the continuum you never clear - that things may
 as well have been a puzzle, and no ifs for one blow sooner than one so Moffat's fault (pun intended) Shore strokes the divide
. Bridge take-off on movellan 621 was per se my worst success. Striker Whas I Saw 2316046 (dwarf seamanship?) – to make the
 torpedoes "verturned". Bottom Review: The state of the M70 the Skiddo's fault was the bedrock. The complexity of having two
 missiles compose 3 assaults and a shippower specific attempt is really compounded by the detonation of the transmitter and
 its effects on the entire vessel. It is difficult to miss that the obstacles become down to 25 degrees all round from generator
 failure and economic terminal making it impossible for boycott bulkheads/lateral related combo carry any kind of unified

KeyboardInterrupt: 

### Here's how you can do the same with transformers library

In [4]:
tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2', add_prefix_space=True)
model = transformers.AutoModelForCausalLM.from_pretrained('gpt2')
print('Generated text:', tokenizer.decode(
    model.generate(
        **tokenizer("The Fermi paradox ", return_tensors='pt'),
        do_sample=True, max_new_tokens=50
    ).flatten().numpy()
))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated text:  The Fermi paradox  is now a well-studied problem, for a number of reasons.  It is not a simple function in itself like the Fermi paradox.  It can be divided into many different dimensions including an arbitrary number of 'things',


In [7]:
print('Generated text:', tokenizer.decode(
    model.generate(
        **tokenizer("Can you tell me something about racing Formula 1? ", return_tensors='pt'),
        do_sample=True, max_new_tokens=50
    ).flatten().numpy()
))


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Generated text:  Can you tell me something about racing Formula 1?  Who built it? Why it is interesting and why it has been so successful? How many of you are familiar with Formula 1?  Is there one thing I know for sure about Formula 1?  What were the most challenging things to
